## 09a — Sanity checks on per-tile enriched CSV

Flags data quality issues in `outputs/scratch/per_tile_enriched_all_cities.csv`
before running notebook 09 statistical analysis.

**Output:** `outputs/scratch/sanity_flags.csv` — fill the `decision` column
(`ok` / `exclude` / `include_with_note`) before running notebook 09.

In [ ]:
# ── Cell 1 — Setup & load ────────────────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from google.colab import drive
drive.mount('/content/drive')

CONFIG_PATH  = Path('/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
# Code from GitHub, data from Drive (see colab_bootstrap.py in the repo).
# Drive PROJECT_ROOT/src is a stale hand-copy; never import from it.
import subprocess as _sp
_sp.run(['wget','-q','-O','/content/colab_bootstrap.py','https://raw.githubusercontent.com/GFDRR/urban_validation/fix/pipeline-audit/colab_bootstrap.py'], check=False)
sys.path.insert(0, '/content')
from colab_bootstrap import setup as _setup
_setup(PROJECT_ROOT)

SCRATCH_DIR = PROJECT_ROOT / 'outputs' / 'scratch'
CSV_PATH    = SCRATCH_DIR / 'per_tile_enriched_all_cities.csv'

df = pd.read_csv(CSV_PATH)

print('=== Dataset overview ===')
print(f'  Total rows      : {len(df):,}')
print(f'  Unique cities   : {df["city"].nunique()}')
print(f'  Unique datasets : {sorted(df["dataset"].unique().tolist())}')
print(f'\n  Columns ({len(df.columns)}):')
for col in df.columns:
    dtype  = str(df[col].dtype)
    n_null = df[col].isna().sum()
    null_s = f'  ({n_null:,} NaN)' if n_null else ''
    print(f'    {col:<45} {dtype}{null_s}')

In [ ]:
# ── Cell 2 — Run all checks ──────────────────────────────────────────────────
# Each row in sanity_flags: city | tile_id | dataset | flag_type | flag_severity | value

flags_list = []

def collect(sub_df, flag_type, severity, value_series):
    """Append flagged rows to flags_list."""
    if sub_df.empty:
        return
    result = sub_df[['city', 'tile_id', 'dataset']].copy().reset_index(drop=True)
    result['flag_type']     = flag_type
    result['flag_severity'] = severity
    result['value']         = value_series.reset_index(drop=True)
    flags_list.append(result)
    print(f'  {flag_type:<30} : {len(result):>6,} rows flagged')

print('=== Row-level checks ===')

# EMPTY_TILE — ref_building_count_centroid == 0
m = df['ref_building_count_centroid'] == 0
collect(df[m], 'EMPTY_TILE', 'medium',
        df.loc[m, 'ref_building_count_centroid'].reset_index(drop=True))

# TINY_BUILDINGS — mean area < 5 m²
m = df['mean_ref_building_area_m2'].notna() & (df['mean_ref_building_area_m2'] < 5)
collect(df[m], 'TINY_BUILDINGS', 'medium',
        df.loc[m, 'mean_ref_building_area_m2'].reset_index(drop=True))

# HUGE_BUILDINGS — mean area > 10 000 m²
m = df['mean_ref_building_area_m2'].notna() & (df['mean_ref_building_area_m2'] > 10_000)
collect(df[m], 'HUGE_BUILDINGS', 'high',
        df.loc[m, 'mean_ref_building_area_m2'].reset_index(drop=True))

# EXTREME_DENSITY — > 5 000 bldg/km²
m = df['ref_building_density_per_km2'].notna() & (df['ref_building_density_per_km2'] > 5_000)
collect(df[m], 'EXTREME_DENSITY', 'high',
        df.loc[m, 'ref_building_density_per_km2'].reset_index(drop=True))

# EDGE_TILE — tile area < 0.5 km² (partial tiles at AOI boundary)
m = df['tile_area_km2'].notna() & (df['tile_area_km2'] < 0.5)
collect(df[m], 'EDGE_TILE', 'low',
        df.loc[m, 'tile_area_km2'].reset_index(drop=True))

# NEAR_PERFECT_F1 — f1 >= 0.99
m = df['f1'].notna() & (df['f1'] >= 0.99)
collect(df[m], 'NEAR_PERFECT_F1', 'medium',
        df.loc[m, 'f1'].reset_index(drop=True))

# EXTREME_BIAS — |signed_area_bias| > 0.5 (only if column present)
if 'signed_area_bias' in df.columns:
    m = df['signed_area_bias'].notna() & (df['signed_area_bias'].abs() > 0.5)
    collect(df[m], 'EXTREME_BIAS', 'high',
            df.loc[m, 'signed_area_bias'].reset_index(drop=True))
else:
    print(f'  {"EXTREME_BIAS":<30} : column signed_area_bias absent — skipped')

# DUPLICATE_ROW — same (city, dataset, tile_id) appears more than once
dup_count = df.groupby(['city', 'dataset', 'tile_id'])['city'].transform('size')
m = df.duplicated(subset=['city', 'dataset', 'tile_id'], keep=False)
collect(df[m], 'DUPLICATE_ROW', 'high',
        dup_count[m].reset_index(drop=True))

print()
print('=== Tile-group checks ===')

# ALL_ZERO_F1 — all datasets have f1 == 0 for the same (city, tile_id)
all_zero = df.groupby(['city', 'tile_id'])['f1'].transform(lambda x: (x == 0).all())
m = all_zero.astype(bool)
collect(df[m], 'ALL_ZERO_F1', 'medium',
        df.loc[m, 'f1'].reset_index(drop=True))

# HIGH_DATASET_DIVERGENCE — max(f1) − min(f1) > 0.4 across datasets for same (city, tile_id)
divergence = df.groupby(['city', 'tile_id'])['f1'].transform(lambda x: x.max() - x.min())
m = divergence > 0.4
collect(df[m], 'HIGH_DATASET_DIVERGENCE', 'medium',
        divergence[m].reset_index(drop=True))

# ── Combine all flags ─────────────────────────────────────────────────────────
if flags_list:
    sanity_flags = pd.concat(flags_list, ignore_index=True)
    sanity_flags['value'] = pd.to_numeric(sanity_flags['value'], errors='coerce').round(4)
else:
    sanity_flags = pd.DataFrame(
        columns=['city', 'tile_id', 'dataset', 'flag_type', 'flag_severity', 'value']
    )

print()
unique_flagged = sanity_flags[['city', 'tile_id', 'dataset']].drop_duplicates().shape[0]
print(f'Total flag entries                    : {len(sanity_flags):,}')
print(f'Unique (city, tile_id, dataset) flagged: {unique_flagged:,} of {len(df):,}')

In [ ]:
# ── Cell 3 — Summary ─────────────────────────────────────────────────────────
SEV_ORDER = {'high': 0, 'medium': 1, 'low': 2}

print('=== Flag counts by type and severity ===')
summary = (
    sanity_flags
    .groupby(['flag_severity', 'flag_type'])
    .size()
    .rename('n_flags')
    .reset_index()
    .sort_values(
        ['flag_severity', 'flag_type'],
        key=lambda col: col.map(SEV_ORDER) if col.name == 'flag_severity' else col
    )
    .reset_index(drop=True)
)
print(summary.to_string(index=False))

print()
print('=== Cities affected by HIGH-severity flags ===')
high_flags = sanity_flags[sanity_flags['flag_severity'] == 'high']
if high_flags.empty:
    print('  None')
else:
    high_city_summary = (
        high_flags
        .groupby('city')['flag_type']
        .apply(lambda x: ', '.join(sorted(x.unique())))
        .rename('high_flags')
        .reset_index()
        .sort_values('city')
    )
    for _, row in high_city_summary.iterrows():
        print(f'  {row["city"]:<35} {row["high_flags"]}')

print()
flagged_rows = sanity_flags[['city', 'tile_id', 'dataset']].drop_duplicates().shape[0]
print(f'Total flagged (city, tile_id, dataset) : {flagged_rows:,} / {len(df):,}'
      f'  ({flagged_rows / len(df) * 100:.1f}%)')

In [ ]:
# ── Cell 4 — City-level building count mismatch ───────────────────────────────
# ref_building_count_centroid derives from the reference dataset, so summing across
# all tiles should give the same total regardless of which candidate dataset the
# row belongs to. A discrepancy > 20% between per-dataset sums signals a join
# artefact, tile-grid mismatch, or duplicate tiles being counted per dataset.

city_ds_sums = (
    df.groupby(['city', 'dataset'])['ref_building_count_centroid']
    .sum()
    .reset_index()
)

city_stats = (
    city_ds_sums
    .groupby('city')['ref_building_count_centroid']
    .agg(min_sum='min', max_sum='max')
    .reset_index()
)
city_stats['pct_diff'] = (
    (city_stats['max_sum'] - city_stats['min_sum'])
    / city_stats['max_sum'].replace(0, np.nan)
)

mismatch = city_stats[city_stats['pct_diff'] > 0.20].sort_values('pct_diff', ascending=False)

print('=== COUNT_MISMATCH — per-dataset building count sum differs > 20% within a city ===')
if mismatch.empty:
    print('  No mismatches found.')
else:
    print(f'  {len(mismatch)} cities flagged:\n')
    # Show the per-dataset breakdown for flagged cities
    flagged_city_names = mismatch['city'].tolist()
    detail = city_ds_sums[city_ds_sums['city'].isin(flagged_city_names)].copy()
    detail = detail.merge(mismatch[['city', 'pct_diff']], on='city')
    detail = detail.sort_values(['pct_diff', 'city', 'dataset'], ascending=[False, True, True])
    detail['pct_diff'] = (detail['pct_diff'] * 100).round(1).astype(str) + '%'
    detail.columns = ['city', 'dataset', 'total_ref_buildings', 'max_vs_min_diff']
    print(detail.to_string(index=False))

print(f'\nTotal cities checked : {len(city_stats)}')
print(f'Cities OK            : {len(city_stats) - len(mismatch)}')
print(f'Cities with mismatch : {len(mismatch)}')

In [ ]:
# ── Cell 5 — Export ───────────────────────────────────────────────────────────
# Add decision column for manual review.
# Fill with: ok | exclude | include_with_note

sanity_flags['decision'] = ''

out_path = SCRATCH_DIR / 'sanity_flags.csv'
sanity_flags.to_csv(out_path, index=False)

print(f'Saved → {out_path}')
print(f'  Rows   : {len(sanity_flags):,}')
print(f'  Columns: {list(sanity_flags.columns)}')
print()
print('=== Flag count summary ===')
print(
    sanity_flags.groupby(['flag_severity', 'flag_type'])
    .size()
    .rename('n_flags')
    .reset_index()
    .sort_values(
        ['flag_severity', 'flag_type'],
        key=lambda col: col.map(SEV_ORDER) if col.name == 'flag_severity' else col
    )
    .reset_index(drop=True)
    .to_string(index=False)
)
print()
print('=== HIGH-severity cities ===')
if high_flags.empty:
    print('  None')
else:
    print(', '.join(sorted(high_flags['city'].unique())))
print()
print('Next step: run Cell 6 to apply auto-decisions and see which cities need manual review.')

In [ ]:
# ── Cell 6 — Auto-decisions + manual review list ─────────────────────────────
# Applies known-good decisions automatically; prints city/tile counts for
# flags that still need a human call before notebook 09 is run.

# Tiles with no reference buildings are structurally uninformative for
# density analysis — exclude them.
sanity_flags.loc[sanity_flags['flag_type'] == 'EMPTY_TILE', 'decision'] = 'exclude'

# High divergence between datasets is the signal we are studying — keep.
sanity_flags.loc[sanity_flags['flag_type'] == 'HIGH_DATASET_DIVERGENCE', 'decision'] = 'ok'

# All-zero F1 and near-perfect F1 are real data points, not errors — keep.
sanity_flags.loc[
    sanity_flags['flag_type'].isin(['ALL_ZERO_F1', 'NEAR_PERFECT_F1']),
    'decision'
] = 'ok'

# ── Cities needing manual review ──────────────────────────────────────────────
review_flags = ['EXTREME_BIAS', 'EXTREME_DENSITY', 'HUGE_BUILDINGS', 'TINY_BUILDINGS']
city_summary = (
    sanity_flags[sanity_flags['flag_type'].isin(review_flags)]
    .groupby(['city', 'flag_type'])['tile_id']
    .count()
    .reset_index()
    .rename(columns={'tile_id': 'n_flagged_tiles'})
)
print('=== Cities requiring manual decision ===')
if city_summary.empty:
    print('  None — all flags resolved automatically.')
else:
    print(city_summary.to_string(index=False))

# ── Re-save with decisions applied ────────────────────────────────────────────
out_path = SCRATCH_DIR / 'sanity_flags.csv'
sanity_flags.to_csv(out_path, index=False)

undecided = (sanity_flags['decision'] == '').sum()
print(f'\nSaved → {out_path}  ({len(sanity_flags):,} rows)')
decided = sanity_flags[sanity_flags['decision'] != '']['decision'].value_counts().to_dict()
print(f'Decisions set  : {decided}')
print(f'Still undecided: {undecided:,} rows — fill these manually in sanity_flags.csv')